# NB_DEPLOY

Deploys the Stratum framework: 3 workspaces per environment (**Data** /
**Integration** / **Code**), the Landing/Bronze/Gold (+Silver if configured)
lakehouses, the metadata catalog SQL Database, and every item in the Code
workspace -- all downloaded fresh from `schultx/FabricFramework@main` on
every run.

Pure `requests` + `notebookutils` throughout -- **no `ms-fabric-cli`, no
`sempy`**. (`%pip install ms-fabric-cli` was found to silently break
`sempy.fabric`'s context provider for the rest of the Spark session; avoiding
both dependencies sidesteps the bug at the root instead of working around it.)

Safe to re-run: every step is idempotent (create-if-missing / overwrite
content on existing items).

In [ ]:
# Parameters
# deploy/run_notebook.py sets this by editing this cell's default via
# updateDefinition before every run (Fabric's Job Scheduler silently ignores
# job-level `parameters` for RunNotebook jobs -- confirmed live, no error).
target_environments_csv = ""   # "" deploys all three (development, test, production)
target_environments = [e.strip() for e in target_environments_csv.split(",") if e.strip()]


In [ ]:
import base64
import io
import json
import struct
import time
import zipfile

import requests
import yaml

FABRIC_API = "https://api.fabric.microsoft.com/v1"
GITHUB_REPO = "schultx/FabricFramework"
GITHUB_BRANCH = "main"


def _token(resource: str = "pbi") -> str:
    return notebookutils.credentials.getToken(resource)


def fabric_headers() -> dict:
    return {"Authorization": f"Bearer {_token('pbi')}", "Content-Type": "application/json"}


def storage_headers() -> dict:
    return {"Authorization": f"Bearer {_token('storage')}"}


REQUEST_TIMEOUT = 60  # seconds -- a hung request with no timeout blocks the whole
                      # notebook session indefinitely; Fabric then kills the entire
                      # session ("System cancelled ... statement execution failures")
                      # rather than surfacing a catchable error. Fail fast and retry instead.
MAX_RETRIES = 3


def api(method: str, path: str, **kwargs) -> requests.Response:
    kwargs.setdefault("timeout", REQUEST_TIMEOUT)
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.request(method, f"{FABRIC_API}{path}", headers=fabric_headers(), **kwargs)
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            print(f"    {method} {path} attempt {attempt}/{MAX_RETRIES} raised {exc!r}, retrying")
            time.sleep(5 * attempt)
            continue
        if resp.status_code >= 500 and attempt < MAX_RETRIES:
            print(f"    {method} {path} attempt {attempt}/{MAX_RETRIES} -> {resp.status_code}, retrying")
            time.sleep(5 * attempt)
            continue
        if resp.status_code >= 400:
            raise RuntimeError(f"{method} {path} -> {resp.status_code}: {resp.text[:1000]}")
        return resp
    raise RuntimeError(f"{method} {path} failed after {MAX_RETRIES} attempts: {last_exc!r}")


def poll_lro(resp: requests.Response) -> requests.Response:
    """Poll a long-running Fabric operation (202 + Location header) to completion."""
    if resp.status_code != 202:
        return resp
    location = resp.headers["Location"]
    retry_after = int(resp.headers.get("Retry-After", "5"))
    while True:
        time.sleep(retry_after)
        poll = requests.get(location, headers=fabric_headers(), timeout=REQUEST_TIMEOUT)
        if poll.status_code == 200:
            body = poll.json()
            if body.get("status") in ("Succeeded", "Completed"):
                return poll
            if body.get("status") == "Failed":
                raise RuntimeError(f"Long-running operation failed: {body}")
        elif poll.status_code not in (200, 202):
            poll.raise_for_status()


## Download `src/` and `config/` from git

Same branch/ref every environment deploys from -- no local drift between dev/test/prod.

In [ ]:
zip_url = f"https://github.com/{GITHUB_REPO}/archive/refs/heads/{GITHUB_BRANCH}.zip"
zip_bytes = requests.get(zip_url, timeout=REQUEST_TIMEOUT).content
archive = zipfile.ZipFile(io.BytesIO(zip_bytes))
root_prefix = archive.namelist()[0]  # "FabricFramework-main/"


def read_repo_file(relative_path: str) -> bytes:
    return archive.read(f"{root_prefix}{relative_path}")


def read_repo_text(relative_path: str) -> str:
    return read_repo_file(relative_path).decode("utf-8")


environments_cfg = yaml.safe_load(read_repo_text("config/environments.yaml"))
lakehouses_cfg = yaml.safe_load(read_repo_text("config/lakehouses.yaml"))
items_cfg = yaml.safe_load(read_repo_text("config/items.yaml"))
metadata_sql = read_repo_text("config/metadata_schema.sql")

environments = environments_cfg["environments"]
if target_environments:
    environments = [e for e in environments if e["name"] in target_environments]
workspace_roles = environments_cfg.get("workspace_roles", [])

print(f"Deploying environment(s): {[e['name'] for e in environments]}")


## Workspace + capacity + role helpers

In [ ]:
def get_capacity_id(capacity_name: str) -> str:
    resp = api("GET", "/capacities")
    matches = [c for c in resp.json()["value"] if c["displayName"] == capacity_name]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one capacity named '{capacity_name}', found {len(matches)}")
    return matches[0]["id"]


def get_or_create_workspace(display_name: str, capacity_id: str) -> str:
    resp = api("GET", "/workspaces")
    matches = [w for w in resp.json()["value"] if w["displayName"] == display_name]
    if matches:
        workspace_id = matches[0]["id"]
        print(f"  workspace exists: {display_name}")
    else:
        resp = api("POST", "/workspaces", json={"displayName": display_name})
        workspace_id = resp.json()["id"]
        print(f"  created workspace: {display_name}")

    api("POST", f"/workspaces/{workspace_id}/assignToCapacity", json={"capacityId": capacity_id})

    for role in workspace_roles:
        api("POST", f"/workspaces/{workspace_id}/roleAssignments", json={
            "principal": {"id": role["principal_id"], "type": role["principal_type"]},
            "role": role["role"],
        })

    return workspace_id


def get_or_create_lakehouse(workspace_id: str, display_name: str) -> str:
    resp = api("GET", f"/workspaces/{workspace_id}/items")
    matches = [i for i in resp.json()["value"] if i["type"] == "Lakehouse" and i["displayName"] == display_name]
    if matches:
        print(f"    lakehouse exists: {display_name}")
        return matches[0]["id"]
    resp = api("POST", f"/workspaces/{workspace_id}/items", json={
        "displayName": display_name, "type": "Lakehouse",
    })
    body = poll_lro(resp).json() if resp.status_code == 202 else resp.json()
    print(f"    created lakehouse: {display_name}")
    return body["id"]


## Generic item deploy (Notebook / DataPipeline / VariableLibrary)

Every item type in the Code workspace round-trips through the same shape: a
folder of files becomes a base64 `definitionParts` list, POSTed on first
create or PATCHed via `updateDefinition` on every redeploy.

In [ ]:
ITEM_PART_FILENAMES = {
    "Notebook": ["notebook-content.py"],
    "DataPipeline": ["pipeline-content.json"],
    "VariableLibrary": ["settings.json", "variables.json"],  # valueSets/*.json appended dynamically
}

ITEM_PAYLOAD_PATH = {
    "notebook-content.py": "notebook-content.py",
    "pipeline-content.json": "pipeline-content.json",
    "settings.json": "settings.json",
    "variables.json": "variables.json",
}


def _b64(data: bytes) -> str:
    return base64.b64encode(data).decode("ascii")


def build_definition_parts(item_type: str, folder: str, substitutions: dict) -> list:
    parts = []

    def add_part(payload_path: str, raw: bytes, is_text: bool = True):
        if is_text:
            text = raw.decode("utf-8")
            for token, value in substitutions.items():
                text = text.replace(token, value)
            raw = text.encode("utf-8")
        parts.append({
            "path": payload_path,
            "payload": _b64(raw),
            "payloadType": "InlineBase64",
        })

    add_part(".platform", read_repo_file(f"{folder}/.platform"))

    if item_type == "VariableLibrary":
        add_part("settings.json", read_repo_file(f"{folder}/settings.json"))
        add_part("variables.json", read_repo_file(f"{folder}/variables.json"))
        value_set_names = [n for n in archive.namelist() if n.startswith(f"{root_prefix}{folder}/valueSets/") and n.endswith(".json")]
        for name in value_set_names:
            rel = name[len(root_prefix):]
            add_part(rel[len(f"{folder}/"):], archive.read(name))
    else:
        for filename in ITEM_PART_FILENAMES[item_type]:
            add_part(filename, read_repo_file(f"{folder}/{filename}"))

    return parts


def get_or_create_item(workspace_id: str, item_type: str, display_name: str, folder: str, substitutions: dict) -> str:
    resp = api("GET", f"/workspaces/{workspace_id}/items")
    matches = [i for i in resp.json()["value"] if i["type"] == item_type and i["displayName"] == display_name]
    definition = {"parts": build_definition_parts(item_type, folder, substitutions)}

    if matches:
        item_id = matches[0]["id"]
        resp = api("POST", f"/workspaces/{workspace_id}/items/{item_id}/updateDefinition", json={"definition": definition})
        poll_lro(resp)
        print(f"    updated {item_type}: {display_name}")
    else:
        resp = api("POST", f"/workspaces/{workspace_id}/items", json={
            "displayName": display_name, "type": item_type, "definition": definition,
        })
        body = poll_lro(resp).json() if resp.status_code == 202 else resp.json()
        item_id = body["id"]
        print(f"    created {item_type}: {display_name}")

    return item_id


## Metadata catalog SQL Database

In [ ]:
def get_or_create_sql_database(workspace_id: str, display_name: str) -> tuple:
    resp = api("GET", f"/workspaces/{workspace_id}/items")
    matches = [i for i in resp.json()["value"] if i["type"] == "SQLDatabase" and i["displayName"] == display_name]
    if matches:
        item_id = matches[0]["id"]
        print(f"    SQL database exists: {display_name}")
    else:
        resp = api("POST", f"/workspaces/{workspace_id}/sqldatabases", json={"displayName": display_name})
        body = poll_lro(resp).json() if resp.status_code == 202 else resp.json()
        item_id = body["id"]
        print(f"    created SQL database: {display_name}")

    detail = api("GET", f"/workspaces/{workspace_id}/sqldatabases/{item_id}").json()
    props = detail["properties"]
    return props["serverFqdn"], props["databaseName"]


def run_metadata_schema(server: str, database: str, sql_text: str) -> None:
    import pyodbc

    token_bytes = _token("https://database.windows.net/.default").encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256

    conn_str = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={server},1433;DATABASE={database};Encrypt=yes"
    conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct}, autocommit=True)
    try:
        cursor = conn.cursor()
        # split on a line whose only content (once stripped) is "GO" -- regardless of
        # leading indentation, so callers can pass an indented triple-quoted string too
        batches, current = [], []
        for line in sql_text.splitlines():
            if line.strip() == "GO":
                batches.append("\n".join(current))
                current = []
            else:
                current.append(line)
        if current:
            batches.append("\n".join(current))

        for batch in batches:
            batch = batch.strip()
            if batch:
                cursor.execute(batch)
        print("    metadata schema applied")
    finally:
        conn.close()


## OneLake file upload (demo data seed)

In [ ]:
def upload_file_to_onelake(workspace_id: str, lakehouse_id: str, dest_path: str, data: bytes) -> None:
    base = f"https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lakehouse_id}/Files/{dest_path}"
    headers = storage_headers()

    resp = requests.put(f"{base}?resource=file", headers=headers, timeout=REQUEST_TIMEOUT)
    if resp.status_code not in (201, 200):
        resp.raise_for_status()

    resp = requests.patch(f"{base}?action=append&position=0", headers=headers, data=data, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()

    resp = requests.patch(f"{base}?action=flush&position={len(data)}", headers=headers, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()


## Deploy -- split across several small cells

Each phase loops over every target environment, but stays in its own cell:
headless `RunNotebook` jobs on this runtime appear to enforce a per-statement
execution ceiling well under a minute, and a single cell that did every phase
for every environment (workspaces -> lakehouses -> SQL Database + schema ->
notebooks -> pipelines -> variable library -> demo data) tripped it partway
through, killing the whole session before any exception could even be
caught. Splitting by phase keeps each cell's own work short regardless of how
many environments or items are involved. State that later phases need
(workspace/lakehouse/notebook/pipeline ids) is carried in `env_state`, keyed
by environment name.

In [ ]:
env_state = {env["name"]: {} for env in environments}


### Phase 1 -- workspaces

In [ ]:
for env in environments:
    print(f"\n==================== {env['name']} ====================")
    st = env_state[env["name"]]
    st["include_silver"] = env["include_silver"]
    st["capacity_id"] = get_capacity_id(env["capacity"])
    st["data_ws_name"] = f"Stratum Data ({env['short']})"
    st["integration_ws_name"] = f"Stratum Integration ({env['short']})"
    st["code_ws_name"] = f"Stratum Code ({env['short']})"

    st["data_ws_id"] = get_or_create_workspace(st["data_ws_name"], st["capacity_id"])
    st["integration_ws_id"] = get_or_create_workspace(st["integration_ws_name"], st["capacity_id"])
    st["code_ws_id"] = get_or_create_workspace(st["code_ws_name"], st["capacity_id"])


### Phase 2 -- lakehouses

In [ ]:
for env in environments:
    st = env_state[env["name"]]
    print(f"-- lakehouses ({st['data_ws_name']})")
    st["lakehouse_ids"] = {}
    for lh in lakehouses_cfg["lakehouses"]:
        if lh["always"] or st["include_silver"]:
            st["lakehouse_ids"][lh["name"]] = get_or_create_lakehouse(st["data_ws_id"], lh["name"])


### Phase 3 -- metadata catalog SQL Database + schema

In [ ]:
for env in environments:
    st = env_state[env["name"]]
    print(f"-- metadata catalog ({st['integration_ws_name']})")
    server, database = get_or_create_sql_database(st["integration_ws_id"], "SQL_STRATUM_CATALOG")
    run_metadata_schema(server, database, metadata_sql)
    st["sql_server"] = server
    st["sql_database"] = database


### Phase 4 -- Code workspace notebooks

In [ ]:
for env in environments:
    st = env_state[env["name"]]
    print(f"-- notebooks ({st['code_ws_name']})")
    st["notebook_ids"] = {}
    for item in items_cfg["items"]:
        if item.get("requires_silver") and not st["include_silver"]:
            continue
        if item["type"] != "Notebook":
            continue
        st["notebook_ids"][item["name"]] = get_or_create_item(
            st["code_ws_id"], "Notebook", item["name"], f"src/{item['path']}", substitutions={}
        )


### Phase 5 -- Code workspace pipelines

In [ ]:
for env in environments:
    st = env_state[env["name"]]
    print(f"-- pipelines ({st['code_ws_name']})")
    notebook_ids = st["notebook_ids"]

    pipeline_substitutions = {
        "__NB_LOAD_BRONZE_ID__": notebook_ids["NB_LOAD_BRONZE"],
        "__NB_LOAD_GOLD_ID__": notebook_ids["NB_LOAD_GOLD"],
        "__NB_LIST_LANDING_ENTITIES_ID__": notebook_ids["NB_LIST_LANDING_ENTITIES"],
        "__DATA_WORKSPACE_ID__": st["data_ws_id"],
        "__LANDING_LAKEHOUSE_ID__": st["lakehouse_ids"]["Landing"],
    }
    if st["include_silver"]:
        pipeline_substitutions["__NB_LOAD_SILVER_ID__"] = notebook_ids["NB_LOAD_SILVER"]

    st["pipeline_ids"] = {}
    for item in items_cfg["items"]:
        if item.get("requires_silver") and not st["include_silver"]:
            continue
        if item["type"] != "DataPipeline" or item["name"] == "PL_RUN_ALL":
            continue
        st["pipeline_ids"][item["name"]] = get_or_create_item(
            st["code_ws_id"], "DataPipeline", item["name"], f"src/{item['path']}", pipeline_substitutions
        )

    run_all_substitutions = {
        "__PL_INGEST_SQL_ID__": st["pipeline_ids"]["PL_INGEST_SQL"],
        "__PL_INGEST_FILE_ID__": st["pipeline_ids"]["PL_INGEST_FILE"],
        "__PL_LOAD_BRONZE_ID__": st["pipeline_ids"]["PL_LOAD_BRONZE"],
        "__PL_LOAD_GOLD_ID__": st["pipeline_ids"]["PL_LOAD_GOLD"],
    }
    get_or_create_item(st["code_ws_id"], "DataPipeline", "PL_RUN_ALL", "src/PL_RUN_ALL.DataPipeline", run_all_substitutions)


### Phase 6 -- variable library + demo data seed

In [ ]:
for env in environments:
    st = env_state[env["name"]]
    for item in items_cfg["items"]:
        if item["type"] == "VariableLibrary":
            get_or_create_item(st["code_ws_id"], "VariableLibrary", item["name"], f"src/{item['path']}", substitutions={})

    print(f"-- seeding demo data ({st['data_ws_name']}/Landing)")
    upload_file_to_onelake(
        st["data_ws_id"], st["lakehouse_ids"]["Landing"], "customer/customer.csv", read_repo_file("demodata/customer.csv")
    )
    print(f"Done: {env['name']}")


## Register demo Bronze entity + summary

One-time metadata rows so `NB_LOAD_BRONZE` / `dim_customer` / `fact_signup`
have something to load on first run (the `catalog.*` DDL creates empty
tables only).

In [ ]:
for env in environments:
    st = env_state[env["name"]]
    seed_sql = """
    IF NOT EXISTS (SELECT 1 FROM [catalog].[Connection] WHERE [Name] = 'demo_customer_source')
    INSERT INTO [catalog].[Connection] ([ConnectionGuid], [Name], [Type]) VALUES (NEWID(), 'demo_customer_source', 'FILE')
    GO
    IF NOT EXISTS (SELECT 1 FROM [catalog].[Source] WHERE [Name] = 'demo')
    INSERT INTO [catalog].[Source] ([ConnectionId], [Name], [Namespace])
    SELECT [ConnectionId], 'demo', 'demo' FROM [catalog].[Connection] WHERE [Name] = 'demo_customer_source'
    GO
    IF NOT EXISTS (SELECT 1 FROM [catalog].[LandingEntity] WHERE [SourceObject] = 'customer.csv')
    INSERT INTO [catalog].[LandingEntity] ([SourceId], [SourceObject], [FilePath], [FileType])
    SELECT [SourceId], 'customer.csv', 'customer', 'csv' FROM [catalog].[Source] WHERE [Name] = 'demo'
    GO
    IF NOT EXISTS (SELECT 1 FROM [catalog].[BronzeEntity] WHERE [Name] = 'customer')
    INSERT INTO [catalog].[BronzeEntity] ([LandingEntityId], [Schema], [Name], [PrimaryKeys])
    SELECT [LandingEntityId], 'dbo', 'customer', 'CustomerId' FROM [catalog].[LandingEntity] WHERE [SourceObject] = 'customer.csv'
    GO
    """
    run_metadata_schema(st["sql_server"], st["sql_database"], seed_sql)
    print(f"Seeded demo Bronze entity for {env['name']}")

print("\nDeployment complete for:", [e["name"] for e in environments])
